# Ladder runner on Colab (one GPU)

Runs a campaign built by `tools/colab.py payload <config>` with the same runner as the Kaggle kernels. Runtime: A100 GPU. Secret `KAGGLE_API_TOKEN` must be set. Set `RUN` and `MODEL`, then run all cells.

In [ ]:
RUN = "colab_smoke"
MODEL = "gemma-4-e4b-it-qat-w4a16-ct"
OWNER = "d410qc"
import os, sys, glob, json, subprocess, pathlib, time
from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
# the harness wheelhouse is built for CPython 3.12 (cp312 wheels), as on the Kaggle image
PY = "/content/py312/bin/python"
if not os.path.exists(PY):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "venv", "-q", "-p", "3.12", "--seed", "/content/py312"], check=True)
    r = subprocess.run(["uv", "pip", "install", "-q", "-p", PY, "vllm==0.19.1"], capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
print(subprocess.run([PY, "-c", "import sys, torch, vllm; print(sys.version.split()[0], torch.__version__, torch.cuda.is_available(), vllm.__version__)"],
                     capture_output=True, text=True))

In [ ]:
I = pathlib.Path("/kaggle/input"); I.mkdir(parents=True, exist_ok=True)
def sh(*a):
    print("$", " ".join(a)); r = subprocess.run(a, capture_output=True, text=True)
    print(r.stdout[-600:], r.stderr[-600:]); r.check_returncode()
if not glob.glob(str(I / "comp" / "**" / "tasks.jsonl"), recursive=True):
    sh("kaggle", "competitions", "download", "gemma-4-developer-agent", "-p", "/content/comp_zip")
    sh("unzip", "-q", "-o", glob.glob("/content/comp_zip/*.zip")[0], "-d", str(I / "comp"))
for ds in ["metric/gemma-4-developer-agent-wheelhouse", "zbigniewkirchner/gemma-4-agent-local-eval-wheels",
           "zbigniewkirchner/gemma-4-agent-local-eval-pythons"]:
    d = I / ds.split("/")[1]
    if not d.exists():
        sh("kaggle", "datasets", "download", ds, "-p", str(d), "--unzip")
m = I / "models" / MODEL / "2"
if not glob.glob(str(m / "**" / "config.json"), recursive=True):
    m.mkdir(parents=True, exist_ok=True)
    sh("kaggle", "models", "instances", "versions", "download", f"google/gemma-4/other/{MODEL}/2", "-p", str(m), "--untar")
print(sorted(str(p.relative_to(I)) for p in I.iterdir()))

In [ ]:
P = pathlib.Path("/content/payload")
sh("kaggle", "datasets", "download", f"{OWNER}/g4-colab-payloads", "-p", str(P), "--unzip", "--force")
for z in P.rglob("*.zip"):
    subprocess.run(["unzip", "-q", "-o", str(z), "-d", str(z.with_suffix(""))])
cands = sorted(p for p in P.rglob("run.py") if RUN in str(p)) or sorted(P.rglob("run.py"))
RUNPY = cands[0]; print("payload", RUNPY)
W = pathlib.Path("/kaggle/working"); W.mkdir(parents=True, exist_ok=True)
LOG = W / f"{RUN}.log"
proc = subprocess.Popen([PY, str(RUNPY)], cwd=str(W), stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
print("started", proc.pid)

In [ ]:
# rerun this cell to follow the run; it returns when the runner exits
while proc.poll() is None:
    time.sleep(60)
    print(time.strftime("%H:%M:%S"), LOG.read_text()[-300:].splitlines()[-1:] if LOG.exists() else "")
print("exit", proc.returncode); print(LOG.read_text()[-3000:])

In [ ]:
# upload /kaggle/working as a private dataset for tools/colab.py pull
slug = f"g4-colab-out-{RUN.replace('_', '-')}"
(W / "dataset-metadata.json").write_text(json.dumps({"title": slug, "id": f"{OWNER}/{slug}", "licenses": [{"name": "other"}]}))
sh("kaggle", "datasets", "create", "-p", str(W), "--dir-mode", "zip")